In [0]:
# =====================================================
# PRIMARY PRODUCT REVIEW RATING HISTOGRAM
# =====================================================

rating_histogram_path = (
    "gs://market-intelligence-raw/ebay/"
    "item_details__primary_product_review_rating__rating_histograms/"
)

rating_histogram_df = (
    spark.read
    .option("multiLine", "false")
    .json(rating_histogram_path + "*.jsonl.gz")
)

print("Total rows:", rating_histogram_df.count())
print("Total columns:", len(rating_histogram_df.columns))

rating_histogram_df.printSchema()

display(rating_histogram_df.limit(10))

In [0]:
# =====================================================
# Rating histogram cardinality per listing
# =====================================================

from pyspark.sql import functions as F

rating_counts = (
    rating_histogram_df
    .groupBy("_dlt_parent_id")
    .agg(
        F.count("*").alias("rating_records")
    )
)

display(
    rating_counts
    .agg(
        F.count("*").alias("parent_count"),
        F.avg("rating_records").alias("avg_records"),
        F.expr("percentile_approx(rating_records, 0.25)").alias("p25"),
        F.expr("percentile_approx(rating_records, 0.50)").alias("median"),
        F.expr("percentile_approx(rating_records, 0.75)").alias("p75"),
        F.min("rating_records").alias("min_records"),
        F.max("rating_records").alias("max_records")
    )
)

In [0]:
# =====================================================
# Rating bucket validation
# =====================================================

display(
    rating_histogram_df
    .groupBy("rating")
    .agg(
        F.count("*").alias("record_count"),
        F.countDistinct("_dlt_parent_id").alias("listing_count")
    )
    .orderBy("rating")
)

In [0]:
# =====================================================
# Validate rating histogram → Item Details relationship
# =====================================================

item_details_path = (
    "gs://market-intelligence-raw/ebay/item_details/"
)

item_details_df = (
    spark.read
    .option("multiLine", "false")
    .json(item_details_path + "*.jsonl.gz")
)

rating_parent_ids = (
    rating_histogram_df
    .select(F.col("_dlt_parent_id").alias("parent_dlt_id"))
    .distinct()
)

item_parent_ids = (
    item_details_df
    .select(F.col("_dlt_id").alias("parent_dlt_id"))
    .distinct()
)

matched_parents = (
    rating_parent_ids
    .join(item_parent_ids, on="parent_dlt_id", how="inner")
    .count()
)

unmatched_parents = (
    rating_parent_ids
    .join(item_parent_ids, on="parent_dlt_id", how="left_anti")
    .count()
)

print("Rating histogram parent IDs:", rating_parent_ids.count())
print("Matched to current Item Details:", matched_parents)
print("Not found in current Item Details:", unmatched_parents)

In [0]:
# =====================================================
# Rating histogram — count quality
# =====================================================

rating_histogram_df.agg(
    F.sum(
        F.when(F.col("count").isNull(), 1).otherwise(0)
    ).alias("count_null"),
    F.sum(
        F.when(F.col("count") < 0, 1).otherwise(0)
    ).alias("negative_count"),
    F.min("count").alias("min_count"),
    F.max("count").alias("max_count")
).show()

In [0]:
# =====================================================
# Blank / whitespace-only string check
# =====================================================

string_columns = ["rating"]

blank_profile = rating_histogram_df.agg(*[
    F.sum(
        F.when(
            F.col(c).isNotNull() &
            (F.trim(F.col(c)) == ""),
            1
        ).otherwise(0)
    ).alias(c)
    for c in string_columns
])

blank_counts = blank_profile.first().asDict()

if any(v > 0 for v in blank_counts.values()):
    display(
        spark.createDataFrame(
            [(k, int(v)) for k, v in blank_counts.items() if v > 0],
            ["column_name", "blank_count"]
        )
    )
else:
    print("No empty or whitespace-only string values found.")

# `primary_product_review_rating__rating_histograms` — Complete Profiling Summary

## 1. Schema

### Current Source Schema

```text
_dlt_id        string
_dlt_list_idx  long
_dlt_parent_id string
count          long
rating         string
```

The important schema finding is that `rating` is currently stored as a string, even though the domain clearly represents numeric rating buckets.

Observed rating values are:

- "1"
- "2"
- "3"
- "4"
- "5"

This is a source schema / type-quality issue, not corrupted data. The actual values are valid 1–5 rating buckets.

### 2. Grain

The table has a very clean and deterministic grain:
```text
1 Item Details listing
        ↓
5 histogram records
        ↓
rating buckets: 1, 2, 3, 4, 5
```

- Current Snapshot

| Metric | Result |
|---|---:|
| Total rows | 33,195 |
| Distinct parents | 6,639 |
| Rows per parent | 5 |

The relationship is exact:

- 6,639 × 5 = 33,195

Therefore, this is not an arbitrary child table. It is a fixed five-row rating histogram per listing.

### 3. Rating Bucket Integrity

Every parent contains exactly the expected five rating buckets.

| Rating | Records | Listings |
|---:|---:|---:|
| 1 | 6,639 | 6,639 |
| 2 | 6,639 | 6,639 |
| 3 | 6,639 | 6,639 |
| 4 | 6,639 | 6,639 |
| 5 | 6,639 | 6,639 |

**Findings**
- ✅ No missing rating buckets
- ✅ No unexpected rating values
- ✅ Exactly five records per listing
- ✅ Every rating bucket is represented for every parent
This means the histogram structure itself is structurally complete.

A zero count value does not mean the rating bucket is missing. It means the listing has zero reviews in that particular rating bucket, which is a valid business value.

# 4. Parent Integrity

| Validation | Result |
|---|---:|
| Rating histogram parent IDs | 6,639 |
| Matched to Item Details | 6,639 |
| Not found | 0 |

Every rating histogram belongs to a current Item Details record.

There are therefore no orphan histogram records requiring removal or special handling in Silver.

### 5. count Quality

The count field represents the number of reviews belonging to each individual rating bucket.

| Metric | Result |
|---|---:|
| NULL counts | 0 |
| Negative counts | 0 |
| Minimum | 0 |
| Maximum | 2,854 |

The field is therefore numerically clean.

The minimum value of 0 is legitimate because a listing can have no reviews in a particular rating bucket.

For example:
```text
Rating 1 → 0 reviews
Rating 2 → 2 reviews
Rating 3 → 5 reviews
Rating 4 → 18 reviews
Rating 5 → 125 reviews
```

The zero should therefore be preserved as a real business value rather than converted to `NULL`.


### 6. String 
The profiling found:
`Empty / whitespace-only values`: 0


There are no empty or whitespace-only string values in the profiled string fields.

Combined with the valid 1–5 rating values, this confirms that the string representation of rating is clean and only requires type normalization, not value cleaning.